# 🧪 BlogMS — Test Configuration (`conftest.py`)

ملف إعداد الـ Tests باستخدام **pytest** و **FastAPI TestClient**.

---

### ✨ المحتويات:
- **Test Database**: SQLite مؤقتة بتتعمل وبتتمسح مع كل test
- **DB Override**: بيستبدل الـ production DB بـ test DB تلقائياً
- **Fixtures**: `setup_db` · `client` · `admin_token` · `author_token` · `reader_token`

## 📦 تثبيت المتطلبات

In [ ]:
%pip install pytest fastapi httpx sqlalchemy

## ⚙️ إعداد قاعدة البيانات للـ Tests

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from app.database import Base, get_db

TEST_DATABASE_URL = "sqlite:///./test_blog.db"

engine = create_engine(
    TEST_DATABASE_URL,
    connect_args={"check_same_thread": False}
)

TestingSessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

## 🔄 Override الـ DB Dependency

In [ ]:
from app.main import app

def override_get_db():
    """بيستبدل الـ production DB بـ test DB في كل الـ requests."""
    db = TestingSessionLocal()
    try:
        yield db
    finally:
        db.close()

app.dependency_overrides[get_db] = override_get_db

## 🏗️ Fixture: `setup_db`

In [ ]:
import pytest

@pytest.fixture(scope="function", autouse=True)
def setup_db():
    """بيعمل الجداول قبل كل test وبيمسحها بعده — autouse=True يشغّله تلقائياً."""
    Base.metadata.create_all(bind=engine)
    yield
    Base.metadata.drop_all(bind=engine)

## 🌐 Fixture: `client`

In [ ]:
from fastapi.testclient import TestClient

@pytest.fixture
def client():
    """بيرجع FastAPI TestClient جاهز للاستخدام في الـ tests."""
    return TestClient(app)

## 🔐 Fixture: `admin_token`

In [ ]:
@pytest.fixture
def admin_token(client):
    """Register → Set ADMIN في الـ DB → Login → Token."""
    client.post("/auth/register", json={
        "username": "adminuser", "email": "admin@test.com", "password": "admin123"
    })
    db = TestingSessionLocal()
    from app.models.user import User, UserRole
    user = db.query(User).filter(User.username == "adminuser").first()
    user.role = UserRole.ADMIN
    db.commit()
    db.close()
    resp = client.post("/auth/login", data={"username": "adminuser", "password": "admin123"})
    return resp.json()["access_token"]

## 🔐 Fixture: `author_token`

In [ ]:
@pytest.fixture
def author_token(client):
    """Register → Set AUTHOR في الـ DB → Login → Token."""
    client.post("/auth/register", json={
        "username": "authoruser", "email": "author@test.com", "password": "author123"
    })
    db = TestingSessionLocal()
    from app.models.user import User, UserRole
    user = db.query(User).filter(User.username == "authoruser").first()
    user.role = UserRole.AUTHOR
    db.commit()
    db.close()
    resp = client.post("/auth/login", data={"username": "authoruser", "password": "author123"})
    return resp.json()["access_token"]

## 🔐 Fixture: `reader_token`

In [ ]:
@pytest.fixture
def reader_token(client):
    """Register (default READER role) → Login → Token."""
    client.post("/auth/register", json={
        "username": "readeruser", "email": "reader@test.com", "password": "reader123"
    })
    resp = client.post("/auth/login", data={"username": "readeruser", "password": "reader123"})
    return resp.json()["access_token"]

## ▶️ تشغيل الـ Tests

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)